In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.orders
(
    order_id INT,
    customer_id INT,

    order_date TIMESTAMP,
    order_status STRING,

    total_amount DECIMAL(12,2),
    payment_status STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("de_learning_workspace.bronze.orders")

silver_df = (
    bronze_df

    .dropDuplicates(["order_id"])

    .withColumn(
        "order_status",
        trim(col("order_status"))
    )

    .withColumn(
        "payment_status",
        trim(col("payment_status"))
    )

    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_learning_workspace.silver.orders")
)